<a href="https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Prakritibhandari07/FlyRank-ml-internship"
REPO_DIR = "FlyRank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo/path"
print("Starter data found. You're ready.")

Working dir: /content/FlyRank-ml-internship
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is primarily a **classification** task, used to produce a **ranking**.

I'm predicting whether a page is declining (a binary label), but the
real output isn't just "yes/no" — it's a ranked list ordered by predicted
probability, so a reviewer can work down the list starting with the most
likely candidates. Classification produces the score; ranking is how
that score gets used.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label`, defined as `trend_direction == "down"`.

This is a **proxy label**, not a clean future outcome — it's a bucket
calculated from the current window, not something that happens after a
defined decision point. A stronger version (planned for later weeks)
would use a future-looking label instead, like "declined over the next
30 days" measured after a feature window — this avoids the risk of the
label just describing the present rather than predicting anything.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

I'm defending this over plain accuracy because a reviewer only has
capacity to check a limited number of pages — the top 50, in this case.
What matters is: of the top 50 pages the model flags, how many are
genuinely worth reviewing? That directly matches how the output gets
used, unlike accuracy, which would be misleading here since the dataset
is close to balanced anyway (54% declining) and doesn't reflect
real-world reviewer capacity.

In [2]:
print(f"Baseline rule Precision@50: 0.240 (~12 of top 50 correct)")
print(f"Random forest Precision@50: 0.740 (~37 of top 50 correct)")

Baseline rule Precision@50: 0.240 (~12 of top 50 correct)
Random forest Precision@50: 0.740 (~37 of top 50 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (`content_id`). Loading the starter slice
below to show this concretely.

In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print(f"Unique content_id values: {df['content_id'].nunique()}")
df[["content_id", "impressions_90d", "days_since_last_update", "trend_direction"]].head(5)

Rows: 30000 | Columns: 44
Unique content_id values: 30000


,content_id,impressions_90d,days_since_last_update,trend_direction
0,content_304f48230142,3803,20,down
1,content_a1fb4e703a9e,15320,25,down
2,content_9aa793d4d895,12581,20,down
3,content_331d6c4de07b,11751,22,stable
4,content_d99b7a2d90ca,19140,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement can only combine a small, fixed number of signals
in a rigid way — e.g., "stale AND visible." It can't learn *how much*
each signal should matter relative to the others, or find non-obvious
interactions between them (like freshness mattering differently depending
on impression volume).

This is exactly what the Week 1 comparison showed: the hand-written
stale-x-visible rule reached Precision@50 of 0.240, while a random forest
reached 0.740 — roughly 3x better — because it could weigh multiple
signals (position, CTR, freshness, word count, engagement) together
instead of relying on one rigid combination a human guessed at.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.